# Histogram and Statistics Table Examples

In this notebook, we utilize concatenated data from the Pacific hake survey to generate histograms and statistics tables under both `echogram-control` mode and `track-control` mode. This data has been seamlessly integrated with geographical coordinates.

## Significance of Histogram and Statistics Table Visualization

Here's why histograms and statistics tables are vital for fisheries scientists:

- Fisheries scientists can use these tools to focus on specific portions of an echogram or sections of a track while scrolling through data. This allows them to examine the distribution of Sv (volume backscattering strength) and gain insights into the types of fish observed.

- For a more comprehensive analysis, fisheries scientists can compare Sv distributions across multiple echosounder channels (frequencies). This comparison helps them determine the likely composition of fish aggregations, providing valuable insights into the ecosystem.

## Import Packages and Data

In [1]:
import panel as pn
import xarray as xr

from echoshader.app import get_box_plot, get_box_stream

pn.extension('bokeh', comms='default')

In [2]:
from urllib import request

# Calibratd data is stored in Google Drive
url = 'https://drive.google.com/uc?export=download&id=197D0MW-bHaF6mZLcQwyr4zqyEHIfwsep'

def urllib_download():
    request.urlretrieve(url, 'concatenated_MVBS.nc')

urllib_download() 

# Load sample data for testing
MVBS_ds = xr.open_mfdataset(
    paths="concatenated_MVBS.nc",
    data_vars="minimal",
    coords="minimal",
    combine="by_coords",
)

MVBS_ds

<xarray.Dataset> Size: 4MB
Dimensions:            (channel: 4, ping_time: 875, echo_range: 150)
Coordinates:
  * channel            (channel) <U37 592B 'GPT  18 kHz 009072058c8d 1-1 ES18...
  * ping_time          (ping_time) datetime64[ns] 7kB 2017-07-24T19:30:00 ......
    time1              (ping_time) datetime64[ns] 7kB dask.array<chunksize=(875,), meta=np.ndarray>
  * echo_range         (echo_range) float64 1kB 0.0 5.0 10.0 ... 740.0 745.0
Data variables:
    Sv                 (channel, ping_time, echo_range) float64 4MB dask.array<chunksize=(4, 875, 150), meta=np.ndarray>
    frequency_nominal  (channel) float64 32B dask.array<chunksize=(4,), meta=np.ndarray>
    longitude          (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
    latitude           (ping_time) float64 7kB dask.array<chunksize=(875,), meta=np.ndarray>
Attributes:
    processing_software_name:     echopype
    processing_software_version:  0.7.1
    processing_time:              2023-05-30T17:40:45Z
    processing_function:          commongrid.compute_MVBS

## Histogram and Table Demonstration in Echogram-Control Mode

Users have the flexibility to tailor the settings for histograms in the following ways:

- `bins`: This parameter allows you to specify the number of bins for the histogram.

- `overlay`: By default, this setting is True, enabling the overlay of multiple histograms. However, you can set it to False to arrange multiple histograms vertically.

In the example below, while in echogram-control mode, users can select a specific area on the echogram, and the corresponding histogram and table will be displayed accordingly.

In [3]:
eg = MVBS_ds.eshader.echogram(
    channel="GPT  18 kHz 009072058c8d 1-1 ES18-11",
).opts(
    cmap="jet",
    clim=(-80, -30),
    colorbar=True,
    tools=["box_select", "hover"],
    width=1250,
    height=450,
)

box_stream = get_box_stream(eg)
box_overlay = get_box_plot(box_stream)

def data_from_box(ds, bounds, vert_dim="echo_range"):
    if bounds is None:
        return ds

    left, bottom, right, top = bounds

    return ds.sel(
        ping_time=slice(left, right),
        **{vert_dim: slice(min(bottom, top), max(bottom, top))},
    )

@pn.depends(box_stream.param.bounds)
def selected_stats(bounds):
    selected_ds = data_from_box(MVBS_ds, bounds)

    hist = selected_ds.eshader.hist(
        bins=20,
        overlay=True,
    )

    table = selected_ds.eshader.table()

    return pn.Column(hist, table)

pn.Column(eg * box_overlay, selected_stats)

Column
    [0] HoloViews(DynamicMap, height=450, sizing_mode='fixed', width=1250)
    [1] ParamFunction(function, _pane=Column, defer_load=False)

There are two controls associated with the histogram. In the refactored architecture, these are explicit Panel widgets created in the notebook rather than state stored on the xarray accessor:

- `bin_size_input`: This widget allows you to adjust the histogram bin size using an input field.

- `overlay_layout_toggle`: This toggle widget provides an option for an overlay layout setting.

In [4]:
bin_size_input = pn.widgets.IntInput(
    name="Bin Size",
    value=20,
    start=1,
)

overlay_layout_toggle = pn.widgets.Checkbox(
    name="Overlay",
    value=True,
)

def histogram_view(bins, overlay):
    return MVBS_ds.eshader.hist(
        bins=bins,
        overlay=overlay,
    )

histogram_bound = pn.bind(
    histogram_view,
    bins=bin_size_input,
    overlay=overlay_layout_toggle,
)

stats_panel = pn.Row(
    pn.Column(
        bin_size_input,
        overlay_layout_toggle,
    ),
    histogram_bound,
)

stats_panel

Row
    [0] Column
        [0] IntInput(label='Bin Size', name='Bin Size', start=1, value=20)
        [1] Checkbox(label='Overlay', name='Overlay', value=True)
    [1] ParamFunction(function, _pane=HoloViews, defer_load=False)

## Histogram and Table Demonstration in Track-Control Mode

The track visualization is now a reusable plotting element without accessor-owned control state. Linking a geographic selection on the track to histograms and tables is handled externally by the notebook/App interaction layer. The example below keeps the track and histogram elements separate while that interaction wiring remains outside the accessor.

In [5]:
track = MVBS_ds.eshader.track(
    tile="OSM",
)

track_hist = MVBS_ds.eshader.hist(
    bins=20,
    overlay=True,
)

pn.Column(
    track,
    track_hist,
)

Column
    [0] HoloViews(Overlay, height=300, sizing_mode='fixed', width=300)
    [1] HoloViews(NdOverlay, height=300, sizing_mode='fixed', width=700)

## Applying Plot Customizations

Similar to customizing echograms, users can input `Holoviews options` to apply personalized adjustments to the visualizations.

For detailed information about `Holoviews options`, please refer to this [link](https://holoviews.org/user_guide/Applying_Customizations.html#option-list-syntax).